# Production normalization standards

This notebook demonstrates the same train-only normalization contract across tabular, image, text, and time-series data. It uses synthetic data and writes no files.

In [1]:
import numpy as np
import pandas as pd
from autoprepml import (
    TabularNormalizer,
    TextPrepML,
    TimeSeriesPrepML,
    fit_image_statistics,
    normalize_image_array,
)

rng = np.random.default_rng(42)

## Tabular data

Fit on training rows only. Identifiers, boolean flags, categories, and targets remain unchanged.

In [2]:
train = pd.DataFrame({
    "temperature": [10.0, 12.0, 14.0, 16.0],
    "yield_kg": [100.0, 120.0, 110.0, 130.0],
    "farm_id": pd.Series(["A", "B", "A", "C"], dtype="string"),
    "target": [0, 1, 0, 1],
})
future = pd.DataFrame({
    "temperature": [18.0], "yield_kg": [150.0],
    "farm_id": pd.Series(["new"], dtype="string"), "target": [0],
})
normalizer = TabularNormalizer(method="robust", columns=["temperature", "yield_kg"])
train_scaled = normalizer.fit_transform(train)
future_scaled = normalizer.transform(future)
print({"fit_rows": len(train), "columns": normalizer.columns_, "future_temperature": round(float(future_scaled.loc[0, "temperature"]), 3)})

{'fit_rows': 4, 'columns': ['temperature', 'yield_kg'], 'future_temperature': 1.667}


## Image data

Pixel values are converted to float values in the 0 through 1 range. Standard mode then applies explicit training-set channel statistics.

In [3]:
pixels = np.array([[[0, 128, 255]], [[32, 64, 96]]], dtype=np.uint8)
mean, std = fit_image_statistics(pixels, channel_axis=-1)
image_values = normalize_image_array(
    pixels, mode="standard",
    mean=mean, std=std, channel_axis=-1,
)
print({"dtype": str(image_values.dtype), "shape": image_values.shape, "finite": bool(np.isfinite(image_values).all())})

{'dtype': 'float32', 'shape': (2, 1, 3), 'finite': True}


## Text data

Unicode NFKC normalization is applied before cleaning while meaningful punctuation is preserved by default.

In [4]:
reviews = pd.DataFrame({"review": ["Ｆｒｅｓｈ potato!", "Good yield <b>today</b>"]})
text_prep = TextPrepML(reviews, text_column="review")
clean_reviews = text_prep.clean_text(remove_html=True)
print(clean_reviews["review"].tolist())

['fresh potato!', 'good yield today']


## Time series

Chronological training rows define the scaler. Future spikes do not change the historical statistics.

In [5]:
series = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=5, freq="D"),
    "yield_kg": [10.0, 12.0, 11.0, 13.0, 1000.0],
})
ts_prep = TimeSeriesPrepML(series, timestamp_column="date", value_column="yield_kg")
normalized_series = ts_prep.fit_transform_normalized(fit_end=4)
print({"fit_rows": 4, "future_value_scaled": round(float(normalized_series.iloc[-1]["yield_kg"]), 3)})

{'fit_rows': 4, 'future_value_scaled': 884.141}


The fitted objects and their recorded columns and statistics should be persisted with the model. Refit only when the training data contract changes.